<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 14 · KNN và khoảng cách</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Mô hình không học tham số: hỏi k láng giềng gần nhất. Chọn k, chuẩn hóa, lời nguyền số chiều, và KNN cho hồi quy.</p>
</div>

### Sau notebook này bạn sẽ

- Tính tay KNN cho một điểm mới.
- Chọn k bằng cross-validation như một núm vặn bias-variance.
- Thấy vì sao KNN bắt buộc phải chuẩn hóa và hỏng dần khi nhiều chiều.
- Dùng KNN cho hồi quy và hiểu chi phí của nó.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> dưới 1 phút</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch14-knn/01-y-tuong/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · Hỏi hàng xóm](#trang-1)
2. [Trang 2 · Chọn k](#trang-2)
3. [Trang 3 · Chuẩn hóa và lời nguyền số chiều](#trang-3)
4. [Trang 4 · KNN hồi quy và chi phí](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · Hỏi hàng xóm

Khoảng cách tới 10 bạn và bỏ phiếu.

*Đọc kèm trên web: [Trang 1 · Hỏi hàng xóm](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch14-knn/01-y-tuong/)*

In [ ]:
import numpy as np                                                   # tính toán
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor   # KNN
ten = list("ABCDEFGHIK")                                             # tên 10 bạn
X = np.array([[1, 7], [2, 6], [3, 8], [4, 7], [5, 6], [2, 7], [6, 8], [3, 5], [7, 7], [4, 6]], dtype=float)   # (giờ học, giờ ngủ)
y = (np.array([2, 4, 5, 7, 6, 3.5, 8.5, 4, 9, 6.5]) >= 5).astype(int)    # 1 là đạt
q = np.array([3.4, 6.3])                                             # bạn mới
d = np.sqrt(((X - q) ** 2).sum(axis=1))                              # khoảng cách Euclid
thu_tu = np.argsort(d)                                               # xếp theo khoảng cách
print([(ten[i], float(round(d[i], 4)), int(y[i])) for i in thu_tu[:5]])     # 5 người gần nhất
for k in [1, 3, 5]:                                                  # ba giá trị k
    phieu = y[thu_tu[:k]]                                            # nhãn của k người gần nhất
    print(k, phieu, "đạt" if phieu.mean() > 0.5 else "không đạt")    # bỏ phiếu

Output mong đợi:

```text
[('K', 0.6708, 1), ('D', 0.922, 1), ('H', 1.3601, 0), ('B', 1.4318, 0), ('F', 1.5652, 0)]
1 [1] đạt
3 [1 1 0] đạt
5 [1 1 0 0 0] không đạt
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Cùng bạn mới, k = 3 đoán đạt, k = 5 đoán không đạt: vòng 5 người bắt thêm B và F, cả hai không đạt.</p>
</div>

In [ ]:
for k in [1, 3, 5]:                                         # cùng việc với scikit-learn
    knn = KNeighborsClassifier(n_neighbors=k).fit(X, y)     # fit chỉ là cất dữ liệu
    print(k, knn.predict([q]), knn.predict_proba([q]).round(2))   # nhãn và tỷ lệ phiếu

Output mong đợi:

```text
1 [1] [[0. 1.]]
3 [1] [[0.33 0.67]]
5 [0] [[0.6 0.4]]
```

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Chọn k

Ba giá trị k trên hai vầng trăng, chọn k bằng cross-validation.

*Đọc kèm trên web: [Trang 2 · Chọn k](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch14-knn/02-chon-k/)*

In [ ]:
from sklearn.datasets import make_moons, load_breast_cancer           # hai bộ dữ liệu
from sklearn.model_selection import train_test_split, cross_val_score  # chia và cross-validation
from sklearn.pipeline import make_pipeline                            # ghép bước
from sklearn.preprocessing import StandardScaler                      # chuẩn hóa
Xm, ym = make_moons(n_samples=300, noise=0.3, random_state=0)         # hai nửa vầng trăng
a, b, c_, e = train_test_split(Xm, ym, test_size=0.3, random_state=0) # chia
for k in [1, 15, 100]:                                                # linh hoạt, vừa, cứng
    m = KNeighborsClassifier(k).fit(a, c_)                            # huấn luyện
    print(k, round(m.score(a, c_), 4), round(m.score(b, e), 4))       # train, test

Output mong đợi:

```text
1 1.0 0.8889
15 0.9429 0.9222
100 0.819 0.8222
```

In [ ]:
dd = load_breast_cancer()                                       # ung thư vú, như Chương 12
Xb, yb = dd.data, 1 - dd.target                                 # 1 là ác tính
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.25, random_state=0, stratify=yb)   # cùng cách chia
for k in [1, 3, 5, 11, 21, 51, 151]:                            # nhiều giá trị k
    pl = make_pipeline(StandardScaler(), KNeighborsClassifier(k))   # chuẩn hóa rồi KNN
    s = cross_val_score(pl, Xtr, ytr, cv=5)                     # 5 phần
    print(k, round(pl.fit(Xtr, ytr).score(Xtr, ytr), 4), round(s.mean(), 4))   # train và cross-validation

Output mong đợi:

```text
1 1.0 0.9578
3 0.9859 0.9695
5 0.9765 0.9672
11 0.9742 0.9554
21 0.9554 0.9507
51 0.9413 0.9343
151 0.9038 0.8803
```

<div style="border-left:6px solid #C0392B;background-color:#FBECEA;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#962D22;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">LỖI HAY GẶP</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>Chọn k bằng độ chính xác train.</b> k = 1 luôn được 100% trên train vì mỗi điểm tự tìm thấy chính nó.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Chuẩn hóa và lời nguyền số chiều

Thang đo, chuẩn hóa và lời nguyền số chiều.

*Đọc kèm trên web: [Trang 3 · Chuẩn hóa và lời nguyền số chiều](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch14-knn/03-chuan-hoa-va-so-chieu/)*

In [ ]:
print(Xb[:, [0, 3, 9]].std(axis=0).round(3))                                  # ba đặc trưng, ba thang đo
print(round(KNeighborsClassifier(5).fit(Xtr, ytr).score(Xte, yte), 4))        # không chuẩn hóa
print(round(make_pipeline(StandardScaler(), KNeighborsClassifier(5)).fit(Xtr, ytr).score(Xte, yte), 4))   # có chuẩn hóa

Output mong đợi:

```text
[3.52100e+00 3.51605e+02 7.00000e-03]
0.9231
0.9371
```

In [ ]:
rng = np.random.default_rng(0)                         # seed
for dim in [2, 10, 100, 1000]:                         # số chiều tăng dần
    P = rng.uniform(0, 1, (1000, dim))                 # 1000 điểm trong hình lập phương
    qq = rng.uniform(0, 1, dim)                        # một điểm mới
    dd_ = np.sqrt(((P - qq) ** 2).sum(axis=1))         # khoảng cách tới cả 1000 điểm
    print(dim, round(dd_.min(), 3), round(dd_.max(), 3), round(dd_.min() / dd_.max(), 3))   # gần nhất, xa nhất, tỷ lệ

Output mong đợi:

```text
2 0.01 1.334 0.007
10 0.471 2.111 0.223
100 3.349 4.653 0.72
1000 12.166 13.553 0.898
```

<div style="border-left:6px solid #1F5F8B;background-color:#E9F1F8;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1F5F8B;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">TRỰC GIÁC</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Ở 1000 chiều, láng giềng gần nhất chỉ gần hơn điểm xa nhất khoảng 10%. Mọi điểm đều xa gần bằng nhau, hỏi hàng xóm mất ý nghĩa.</p>
</div>

<a name="trang-4" id="trang-4"></a>

## Trang 4 · KNN hồi quy và chi phí

KNN hồi quy trên bộ tiểu đường.

*Đọc kèm trên web: [Trang 4 · KNN hồi quy và chi phí](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch14-knn/04-hoi-quy-va-chi-phi/)*

In [ ]:
from sklearn.datasets import load_diabetes                     # bộ của Chương 9
from sklearn.metrics import r2_score                           # R²
Xd, yd = load_diabetes(return_X_y=True, as_frame=True, scaled=False)   # đơn vị gốc
Xdtr, Xdte, ydtr, ydte = train_test_split(Xd, yd, test_size=0.2, random_state=42)   # cách chia Chương 9
for k in [1, 5, 20, 50]:                                       # bốn giá trị k
    md = make_pipeline(StandardScaler(), KNeighborsRegressor(k)).fit(Xdtr, ydtr)    # chuẩn hóa rồi KNN hồi quy
    print(k, round(r2_score(ydte, md.predict(Xdte)), 4))       # R² test

Output mong đợi:

```text
1 0.0678
5 0.4248
20 0.4246
50 0.4276
```

<div style="border-left:6px solid #7A6A58;background-color:#F4F1EC;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#5C4E3F;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">THỬ SỬA</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>1.</b> Bỏ StandardScaler trong ô trên. R² thay đổi thế nào?</p>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;"><b>2.</b> Đo thời gian predict bằng %timeit với k = 5 và so với LinearRegression.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Tính tay trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch14-knn/05-bai-tap/)*

### Bài 14.1 · Khoảng cách tay

Tính tay khoảng cách từ bạn mới tới D (4; 7) và H (3; 5).
<details><summary>Xem đáp án</summary>

D: $\sqrt{0{,}85} \approx 0{,}922$. H: $\sqrt{1{,}85} \approx 1{,}3601$. Code ở ô ngay dưới.
</details>

In [ ]:
print(np.sqrt(0.6 ** 2 + 0.7 ** 2), np.sqrt(0.4 ** 2 + 1.3 ** 2))   # D và H

Output mong đợi:

```text
0.9219544457292886 1.3601470508735443
```

### Bài 14.2 · Phiếu có trọng số

Với $k = 5$, cho mỗi láng giềng phiếu nặng $1/d$. Kết quả có đổi không?
<details><summary>Xem đáp án</summary>

Đạt: $2{,}576$; không đạt: $2{,}072$. Kết quả lật thành đạt, xác suất khoảng 0,554. Code ở ô ngay dưới.
</details>

In [ ]:
mw = KNeighborsClassifier(5, weights="distance").fit(X, y)    # phiếu nặng theo nghịch đảo khoảng cách
print(mw.predict([q]), mw.predict_proba([q]).round(4))        # nhãn và xác suất

Output mong đợi:

```text
[1] [[0.4459 0.5541]]
```

### Bài 14.3 · k bằng cả lớp

KNN hồi quy dự đoán điểm của bạn mới với $k = 3$ và $k = 10$.
<details><summary>Xem đáp án</summary>

$k = 3$: $5{,}8333$. $k = 10$: $5{,}55$, trung bình cả lớp, chính là baseline luôn đoán trung bình. Code ở ô ngay dưới.
</details>

In [ ]:
diem = np.array([2, 4, 5, 7, 6, 3.5, 8.5, 4, 9, 6.5])            # điểm thi
for k in [3, 10]:                                               # ba láng giềng, cả lớp
    print(k, KNeighborsRegressor(k).fit(X, diem).predict([q]).round(4))   # trung bình điểm của k láng giềng

Output mong đợi:

```text
3 [5.8333]
10 [5.55]
```